In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb

In [ ]:
train = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/smart_factory/data/raw/train.csv')
test = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/smart_factory/data/raw/test.csv')

In [3]:
train.head()

,ID,layout_id,scenario_id,order_inflow_15m,unique_sku_15m,avg_items_per_order,urgent_order_ratio,heavy_item_ratio,cold_chain_ratio,sku_concentration,...,racking_height_avg_m,cross_dock_ratio,packaging_material_cost,quality_check_rate,outbound_truck_wait_min,dock_to_stock_hours,kpi_otd_pct,backorder_ratio,shift_handover_delay_min,sort_accuracy_pct
0,TRAIN_000000,WH_136,SC_07598,51.0,96.0,3.29,0.1176,0.1765,0.0392,0.3063,...,NaN,NaN,4.60,0.1443,8.1,7.92,86.6,0.0787,5.12,NaN
1,TRAIN_000001,WH_136,SC_07598,NaN,93.0,2.55,0.0597,NaN,0.0149,NaN,...,2.50,0.2490,5.22,0.1400,NaN,5.48,83.9,0.0850,5.77,94.88
2,TRAIN_000002,WH_136,SC_07598,92.0,115.0,2.49,0.0652,0.2283,0.0217,0.3063,...,3.87,0.1977,4.26,0.1817,10.7,6.88,82.1,0.1052,NaN,94.40
3,TRAIN_000003,WH_136,SC_07598,77.0,110.0,2.52,0.0649,NaN,0.0390,0.3063,...,2.50,0.1955,4.89,0.1485,10.7,6.76,87.9,0.0920,4.53,93.72
4,TRAIN_000004,WH_136,SC_07598,NaN,122.0,3.12,0.0667,0.3333,NaN,0.3063,...,NaN,0.2351,5.16,0.1514,12.4,9.03,83.8,0.0843,3.99,95.02


In [9]:
drop_cols = ['ID', 'scenario_id', 'avg_delay_minutes_next_30m','layout_id']
target = 'avg_delay_minutes_next_30m'

In [16]:
X      = train.drop(columns=drop_cols)
y      = train[target]
X_test = test.drop(columns=['ID', 'scenario_id', 'layout_id'])

In [18]:
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_pred  = np.zeros(len(X))
test_pred = np.zeros(len(X_test))
mae_scores = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(
        objective='mae',
        metric='mae',
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=200)
        ]
    )

    val_pred = model.predict(X_val)
    oof_pred[val_idx] = val_pred

    fold_mae = mean_absolute_error(y_val, val_pred)
    mae_scores.append(fold_mae)
    print(f"Fold {fold} MAE: {fold_mae:.4f}")

    test_pred += model.predict(X_test) / N_FOLDS  # fold별 평균

print(f"\n✅ 각 Fold MAE: {[round(s, 4) for s in mae_scores]}")
print(f"✅ 평균 MAE:    {np.mean(mae_scores):.4f}")
print(f"✅ OOF MAE:     {mean_absolute_error(y, oof_pred):.4f}")  # 가장 신뢰도 높음

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.473368 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 19688
[LightGBM] [Info] Number of data points in the train set: 200000, number of used features: 90
[LightGBM] [Info] Start training from score 9.048582
[200]	valid_0's l1: 9.18162
[400]	valid_0's l1: 9.0877
[600]	valid_0's l1: 9.02329
[800]	valid_0's l1: 8.981
[1000]	valid_0's l1: 8.94913
Fold 1 MAE: 8.9491
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.164342 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 19686
[LightGBM] [Info] Number of data points in the train set: 200000, number of used features: 90
[LightGBM] [Info] Start training from score 9.038824
[200]	valid_0's l1: 9.26328
[400]	valid_0's l1: 9.17151
[600]	valid_0's l1: 9.11115
[800]	valid_0's l1: 9.06937
[1000]	valid_0's l1: 8.99401
Fold 2 MAE: 8.

In [19]:
submission = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/smart_factory/sample_submission.csv')
submission['avg_delay_minutes_next_30m'] = test_pred
submission.to_csv('submission_v1_kfold.csv', index=False)
print("\n✅ submission_v1_kfold.csv 저장")



✅ submission_v1_kfold.csv 저장
